In [2]:
from pathlib import Path
import subprocess

ALPHABET : list[str] = ['A', 'C', 'G', 'T']
WORD_LENGTH : int = 8

In [ ]:
def Check_CG_Count(word : str) -> bool:
    """checks the letter count rule for one word"""
    if(word.count('C') + word.count('G') == 4):
        return True
    return False

In [ ]:
def Check_4_Position_Difference(word_x : str, word_y : str) -> bool:
    """checks the 4-pos rule for 2 words"""
    diffCount = 0
    for i in range(WORD_LENGTH):
        if(word_x[i] != word_y[i]):
            diffCount += 1

    if(diffCount >= 4):
        return True
    return False

In [ ]:
def Check_RC_Difference(word_x : str, word_y :str) -> bool:
    """checks reverse vs WCC difference rule for 2 words"""
    R_word = Get_Reverse_Word(word_x)
    C_word = Get_WCC_Word(word_y)

    if(Check_4_Position_Difference(R_word, C_word)):
        return True
    return False

def Get_Reverse_Word(word : str) -> str:
    """returns reverse of a word"""
    return word[::-1]

def Get_WCC_Word(word: str) -> str:
    """returns a word WCC-ed counterpart"""
    complements = {
        'A': 'T',
        'T': 'A',
        'C': 'G',
        'G': 'C',
    }

    return ''.join(complements[letter] for letter in word)

In [ ]:
def GenerateAllEligibleWords() -> list[str]:
    """returns a list of all words that dont break self-RC and CG-count rules"""
    words = [""]
    for i in range(WORD_LENGTH):
        words = GenerateAllEligibleWordsHelper(words)

    return [word for word in words if(Check_CG_Count(word) and Check_RC_Difference(word, word))]

def GenerateAllEligibleWordsHelper(array : list[str]) -> list[str]:
    """helper for creating every possible word creatable form the alphabet"""
    out = []
    for subword in array:
        for i in ALPHABET:
            out.append(subword + i)
    return out

def GenerateVariablesForWords(words : list[str]) -> dict[str , int]:
    """appoints each word a number and returns a dictionary of the mapping"""
    return {word: index for index, word in enumerate(words, start = 1)}

In [ ]:
def GetRuleClauses(words : list[str], wordVariables : dict[str , int]) -> list[tuple[int, int]]:
    """generates all CNF clauses for enforcing the problem rules"""

    clauses = []
    for word_x in words:
        for word_y in words:
            if(word_x == word_y):
                continue
            if(not (Check_4_Position_Difference(word_x, word_y) and Check_RC_Difference(word_x, word_y))):
            # only append if words break the rules to enforce they cannot be true at the same time
                clauses.append((-wordVariables[word_x], -wordVariables[word_y]))

    return clauses

In [ ]:
def GetSizeClauses(wordVariables : dict[str, int], targetSize : int) -> tuple[list[tuple[int, ...]], int]:
    """generates all CNF clauses for enforcing the "exactly k words in set" rule"""
    clauses = []
    selectedVariables = list(wordVariables.values())
    wordCount = len(selectedVariables)
    nextVariable = max(selectedVariables, default = 0) + 1

    # resereve variable for tautology
    trueLiteral = nextVariable
    nextVariable += 1
    clauses.append((trueLiteral,))

    # reserve variable for contradiction
    falseLiteral = -trueLiteral

    # 2D dynamic list of steps
    # dp[i][j] means "atleast j words were selected from the first i words"
    # initial state is false everywhere
    # targetSize + 2 so the encoding can prevent selecting more than targetSize words  
    dp = [[falseLiteral] * (targetSize + 2) for _ in range(wordCount + 1)]

    # set baseline "atleast 0 words in set" as always true
    for i in range(wordCount + 1):
        dp[i][0] = trueLiteral


    for i in range(1, wordCount + 1):
        selectedVariable = selectedVariables[i - 1]

        for j in range(1, targetSize + 2):
            # Create a new auxiliary variable for dp[i][j].
            currentState = nextVariable
            nextVariable += 1
            dp[i][j] = currentState

            # A: at least j words were already selected before word i.
            alreadyReached = dp[i - 1][j]

            # B: at least j - 1 words were selected before word i.
            previousLevel = dp[i - 1][j - 1]

            # main rule is:
            # currentState <-> alreadyReached OR (previousLevel AND selectedVariable)

            # alreadyReached -> currentState
            clauses.append((-alreadyReached, currentState))

            # previousLevel AND selectedVariable -> currentState
            clauses.append((-previousLevel, -selectedVariable, currentState))

            # currentState -> alreadyReached OR (previousLevel AND selectedVariable)
            clauses.append((-currentState, alreadyReached, previousLevel))
            clauses.append((-currentState, alreadyReached, selectedVariable))

    # Require at least targetSize selected words.
    clauses.append((dp[wordCount][targetSize],))

    # Forbid targetSize + 1 or more selected words.
    # Together, these unit clauses require exactly targetSize words.
    clauses.append((-dp[wordCount][targetSize + 1],))

    return clauses, nextVariable - 1


In [ ]:
def Glucose(variableCount : int, clauses : list[tuple[int, ...]], solverPath : str) -> subprocess.CompletedProcess[str]:
    """writes all clauses into a file in the correct glucose format and runs glucose on it"""
    inputPath = Path("glucoseClauses.txt").resolve()
    with inputPath.open('w') as file:
        file.write(f"p cnf {variableCount} {len(clauses)}\n")
        for clause in clauses:
            file.write(f"{' '.join(map(str, clause))} 0\n")

    # required conversion of path from virtual machine linux running glucose
    pathConversion = subprocess.run(
        ["wsl.exe", "--exec", "wslpath", "-a", "-u", str(inputPath)],
        capture_output=True,
        text=True,
        check=True,
    )
    wslInputPath = pathConversion.stdout.strip()

    return subprocess.run(
        ["wsl.exe", "--exec", solverPath, "-model", "-verb=0", wslInputPath],
        capture_output=True,
        text=True,
    )


In [ ]:
def ParseGlucoseModel(result: subprocess.CompletedProcess[str]) -> set[int] | None:
    """parses model and outcome from glucose output"""
    if result.returncode == 20:
        return None

    selectedVariables = set()

    for line in result.stdout.splitlines():
        tokens = line.split()

        if(not tokens or tokens[0] != 'v'):
            continue

        for token in tokens[1:]:
            literal = int(token)

            if(literal == 0):
                break

            if(literal > 0):
                selectedVariables.add(literal)

    return selectedVariables

In [ ]:
def DecodeGlucoseOutput(parsedModel : set[int] | None, setSize : int) -> None:
    """decodes parsed glucose output into a humanly readable string"""
    if(parsedModel is None):
        return print("Glucose didnt find any solutions.")

    selectedWords = [word for word, variable in wordVariables.items() if variable in parsedModel]
    finalSize = len(selectedWords)

    if(finalSize != setSize):
        return print(f"Couldnt find a model with target size.")
    
    return print(f"Found solution for words:\n{'\n'.join(selectedWords)}")
    

In [ ]:
def Run(target_set_size : int, solverPath : str, variableCount : int, clauses : list[tuple[int, ...]]) -> None:
    """connects and runs the whole SAT calculation for a specific set of clauses"""
    glucoseOutput = Glucose(variableCount, clauses, solverPath)
    parsed = ParseGlucoseModel(glucoseOutput)
    DecodeGlucoseOutput(parsed, target_set_size)


In [ ]:
# can be changed for a custom word list
words = GenerateAllEligibleWords()

wordVariables = GenerateVariablesForWords(words)
ruleClauses = GetRuleClauses(words, wordVariables)

# path to solver program
# e.g. "/home/USER/glucose/simp/glucose_release"
solverPath = ""

# 0 <= input set size
set_size = 0

In [ ]:
sizeClauses, variableCount = GetSizeClauses(wordVariables, set_size)
clauses = ruleClauses + sizeClauses

Run(set_size, solverPath, variableCount, clauses)